# IndexTTS 2.5

- **支持显卡**：A100 (40GB / 80GB)、H100 (80GB)、RTX 6000 (96GB)
- **默认配置**：IndexTTS-2.5 BF16（GPT + Zipformer S2Mel + BigVGAN + QwenEmotion）、支持中 / 英 / 日 / 西 / 阿 5 语种零样本克隆、情绪控制与语速调节
- **访问方式**：通过 Colab Secrets 自动读取 `CF_TUNNEL_TOKEN`，挂载 Cloudflare Tunnel 直连自有域名与 API
- **执行顺序**：`1. 环境检测` → `2. 安装 ComfyUI` → `3. 下载模型与工作流` → `4. 启动并打开 ComfyUI`

| 工作流 | 功能 | 输入要求 |
| --- | --- | --- |
| `01_IndexTTS_2_5_Clone` | 基础声音克隆 | 1 段音色参考音频 + 文本（可选填写 `emo_text` 文字情绪描述） |
| `02_IndexTTS_2_5_Emotion_Vector` | 情绪滑块控制克隆 | 1 段音色参考音频 + 8 维情绪滑块（`0~1`）+ 文本 |
| `03_IndexTTS_2_5_Emotion_Ref` | 音色与情绪分离克隆 | 音色参考音频 + 情绪参考音频 + 文本 |


### 1. 环境检测
- **功能**：检测 GPU 型号、显存容量、BF16 支持、系统内存及 PyTorch / CUDA 版本。
- **要求**：支持 `A100`、`H100`、`RTX 6000`，显存 40GB 及以上。
- **使用**：直接运行，输出 `[OK]` 后继续。


In [ ]:
# @title 1. 环境检测 {display-mode: "form"}
import os, torch

if not torch.cuda.is_available():
    raise RuntimeError("未检测到 GPU。请在 Colab 菜单选择：代码执行程序 -> 更改运行时类型 -> A100 / H100 / RTX 6000 GPU。")

name = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
ram_gib = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
bf16 = torch.cuda.is_bf16_supported()
print(f"GPU: {name} | 显存: {vram_gib:.1f} GiB | 系统内存: {ram_gib:.1f} GiB | BF16: {bf16}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")
if not any(k in name.upper() for k in ("A100", "H100", "6000")) or vram_gib < 35:
    raise RuntimeError(f"当前显卡为 {name} ({vram_gib:.1f} GiB)，需要 A100 (40GB/80GB)、H100 (80GB) 或 RTX 6000 (96GB)。")
if not bf16:
    raise RuntimeError("当前 GPU 不支持 BF16。")
print("[OK] GPU 环境符合要求。")


### 2. 安装 ComfyUI
- **功能**：安装固定版本 ComfyUI、`ComfyUI-IndexTTS-2.5` 节点、音频/多语种分词依赖及 `cloudflared`。
- **位置**：`/content/ComfyUI` 与 `/content/ComfyUI/custom_nodes/ComfyUI-IndexTTS-2_5`。
- **使用**：直接运行，首次安装约需 2–3 分钟。


In [ ]:
# @title 2. 安装 ComfyUI {display-mode: "form"}
import re, subprocess, sys, urllib.request
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")
COMFY_REVISION = "c9d8a6e69c4b5ab7fa0f789e7b988c172cd31fc9"
NODE_DIR = COMFY_DIR / "custom_nodes/ComfyUI-IndexTTS-2_5"
NODE_REPO = "https://github.com/mailzwj/ComfyUI-IndexTTS-2.5"
NODE_REVISION = "a501798540c8079530b0e03d824c79763d12e7bd"

def run(args, cwd=None):
    subprocess.run(args, cwd=cwd, check=True)

def checkout_repo(path, repo_url, revision, label):
    if path.is_symlink():
        raise RuntimeError(f"{label} 目录是软链接，请使用新的 runtime。")
    fresh = not path.exists()
    if fresh:
        path.parent.mkdir(parents=True, exist_ok=True)
        run(["git", "clone", "--depth", "1", "--no-checkout", repo_url + ".git", str(path)])
    if not (path / ".git").is_dir():
        raise RuntimeError(f"{label} 目录不是 Git 仓库，请使用新的 runtime。")
    current = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=path, text=True).strip()
    if fresh or current != revision:
        run(["git", "fetch", "--depth", "1", "origin", revision], cwd=path)
        run(["git", "checkout", "--detach", revision], cwd=path)

checkout_repo(COMFY_DIR, "https://github.com/Comfy-Org/ComfyUI", COMFY_REVISION, "ComfyUI")
checkout_repo(NODE_DIR, NODE_REPO, NODE_REVISION, "ComfyUI-IndexTTS-2_5")

# Patch Linux text normalizer in front.py to use prebuilt wetext (avoids compiling pynini)
front_py = NODE_DIR / "indextts/utils/front.py"
front_src = front_py.read_text(encoding="utf-8")
front_src = front_src.replace('if platform.system() != "Linux":', "if True:")
front_py.write_text(front_src, encoding="utf-8")

skip = {"torch", "torchvision", "torchaudio", "xformers"}
lines = []
for req_file in (COMFY_DIR / "requirements.txt", NODE_DIR / "requirements.txt"):
    for line in req_file.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        name = re.split(r"[<>=!~\[]", s, maxsplit=1)[0].strip().lower()
        if name not in skip:
            lines.append(s)

requirements = Path("/content/indextts_comfy_requirements.txt")
requirements.write_text("\n".join(lines) + "\n", encoding="utf-8")
run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(requirements),
    "transformers==4.57.6",
    "wetext>=0.0.9",
    "fugashi>=1.2.0",
    "unidic-lite>=1.0.0",
    "huggingface_hub",
    "hf_xet",
])

cloudflared_bin = Path("/usr/local/bin/cloudflared")
if not cloudflared_bin.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        str(cloudflared_bin),
    )
    cloudflared_bin.chmod(0o755)

INDEXTTS_INSTALLED_REVISIONS = (COMFY_REVISION, NODE_REVISION)
print(f"ComfyUI: {COMFY_REVISION[:12]} | ComfyUI-IndexTTS-2.5: {NODE_REVISION[:12]} | cloudflared: 已就绪")
print("[OK] ComfyUI 与 IndexTTS-2.5 节点安装完成。")


### 3. 下载模型与工作流
- **功能**：下载 `IndexTeam/IndexTTS-2.5` 主权重、辅助编码器/声码器（`w2v-bert-2.0`、`CAMPPlus`、`MaskGCT`、`BigVGAN`）至 `/content/ComfyUI/models/TTS/IndexTTS-2.5`，预置示例参考音频并同步工作流。
- **参数**：
  - `DOWNLOAD_QWEN_EMO`：下载 `qwen0.6bemo4-merge` 文本情绪分析模型（默认开启，支持用文字描述控制情绪）。
  - `WORKFLOW_REF`：工作流 Git 分支或 Commit（默认 `main`）。
- **使用**：直接运行；已存在模型自动跳过，同名工作流直接覆盖。


In [ ]:
# @title 3. 下载模型与工作流 {display-mode: "form"}
DOWNLOAD_QWEN_EMO = True # @param {type:"boolean"}
WORKFLOW_REF = "main" # @param {type:"string"}

import json, os, shutil, sys, tempfile, urllib.error, urllib.request
from pathlib import Path
from urllib.parse import quote
from huggingface_hub import hf_hub_download

COMFY_DIR = Path("/content/ComfyUI")
NODE_DIR = COMFY_DIR / "custom_nodes/ComfyUI-IndexTTS-2_5"
MODEL_DIR = COMFY_DIR / "models/TTS/IndexTTS-2.5"
WORKFLOW_DIR = COMFY_DIR / "user/default/workflows"
INPUT_DIR = COMFY_DIR / "input"
OUTPUT_DIR = COMFY_DIR / "output"

MODEL_REPO = "IndexTeam/IndexTTS-2.5"
MODEL_REVISION = "c39ce5ba981572cb187443877ff559dfb246ce63"

CORE_FILES = [
    "config.yaml",
    "gpt.pth",
    "s2mel.pth",
    "codec.pth",
    "wav2vec2bert_stats.pt",
    "feat1.pt",
    "feat2.pt",
    "multilingual_zh_ja_yue_char_del.tiktoken",
]

QWEN_EMO_FILES = [
    "qwen0.6bemo4-merge/Modelfile",
    "qwen0.6bemo4-merge/added_tokens.json",
    "qwen0.6bemo4-merge/chat_template.jinja",
    "qwen0.6bemo4-merge/config.json",
    "qwen0.6bemo4-merge/generation_config.json",
    "qwen0.6bemo4-merge/merges.txt",
    "qwen0.6bemo4-merge/model.safetensors",
    "qwen0.6bemo4-merge/special_tokens_map.json",
    "qwen0.6bemo4-merge/tokenizer.json",
    "qwen0.6bemo4-merge/tokenizer_config.json",
    "qwen0.6bemo4-merge/vocab.json",
]

WORKFLOW_REPO = "boltguo/colab"
WORKFLOW_FOLDER = "workflows/indextts-2.5"
WORKFLOW_FILES = (
    "01_IndexTTS_2_5_Clone.json",
    "02_IndexTTS_2_5_Emotion_Vector.json",
    "03_IndexTTS_2_5_Emotion_Ref.json",
)

for d in (MODEL_DIR, WORKFLOW_DIR, INPUT_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

files_to_download = list(CORE_FILES) + (QWEN_EMO_FILES if DOWNLOAD_QWEN_EMO else [])
for rel_path in files_to_download:
    target = MODEL_DIR / rel_path
    if target.is_file() and target.stat().st_size > 0:
        print("已存在：", rel_path)
        continue
    hf_hub_download(
        repo_id=MODEL_REPO,
        filename=rel_path,
        revision=MODEL_REVISION,
        local_dir=str(MODEL_DIR),
    )
    print("已下载：", rel_path)

# Pre-download auxiliary models (w2v-bert-2.0, MaskGCT, CAMPPlus, BigVGAN) into hf_cache
if str(NODE_DIR) not in sys.path:
    sys.path.insert(0, str(NODE_DIR))
from indextts.utils.model_download import ensure_models_available
ensure_models_available(str(MODEL_DIR))

# Copy example reference audio into ComfyUI input/
sample_audio = NODE_DIR / "example/ref_audio.mp3"
if sample_audio.is_file() and not (INPUT_DIR / "ref_audio.mp3").exists():
    shutil.copy2(sample_audio, INPUT_DIR / "ref_audio.mp3")
    print("已预置示例参考音频：/content/ComfyUI/input/ref_audio.mp3")

def download_github_workflows():
    ref = WORKFLOW_REF.strip() or "main"
    for name in WORKFLOW_FILES:
        url = f"https://raw.githubusercontent.com/{WORKFLOW_REPO}/{quote(ref, safe='')}/{WORKFLOW_FOLDER}/{quote(name, safe='')}"
        req = urllib.request.Request(url, headers={"User-Agent": "IndexTTS25-Colab-Workflow-Downloader"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        target = WORKFLOW_DIR / name
        with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=WORKFLOW_DIR, delete=False) as tf:
            json.dump(data, tf, ensure_ascii=False, indent=2)
            tf.write("\n")
            tmp_name = tf.name
        os.replace(tmp_name, target)
        print("工作流已安装：", name)

download_github_workflows()
print("[OK] IndexTTS-2.5 模型与工作流准备完成。")


### 4. 启动并打开 ComfyUI
- **功能**：后台启动 ComfyUI（`127.0.0.1:8188`），自动从 Colab Secrets 读取 `CF_TUNNEL_TOKEN` 启动 Cloudflare Tunnel。
- **要求**：在 Colab 左侧 **🔑 Secrets（密钥）** 中添加 `CF_TUNNEL_TOKEN` 并开启访问权限（保存一次即可，无需每次输入）。
- **使用**：直接运行；启动后直接通过你在 Cloudflare 绑定的域名访问 UI 或调用 API。


In [ ]:
# @title 4. 启动并打开 ComfyUI {display-mode: "form"}
import json, os, re, signal, socket, subprocess, sys, time, urllib.request
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")
BASE = "http://127.0.0.1:8188"
LOG = Path("/content/indextts_comfyui.log")
CF_LOG = Path("/content/cloudflared.log")
CF_PID = Path("/content/cloudflared.pid")

def get_json(path):
    with urllib.request.urlopen(f"{BASE}{path}", timeout=5) as response:
        return json.load(response)

def ready():
    try:
        get_json("/system_stats")
        return True
    except Exception:
        return False

def get_listen_addrs():
    try:
        with socket.socket(socket.AF_INET6, socket.SOCK_STREAM) as s:
            s.bind(("::1", 0))
        return "127.0.0.1,::1"
    except OSError:
        return "127.0.0.1"

def stop_idle_comfy():
    try:
        q = get_json("/queue")
        if q.get("queue_running") or q.get("queue_pending"):
            return False
    except Exception:
        pass
    subprocess.run(["pkill", "-f", r"(_comfy_userdata_compat\.py|main\.py.*8188)"], check=False)
    time.sleep(2)
    return True

def resolve_cf_token():
    t = ""
    try:
        from google.colab import userdata
        t = (userdata.get("CF_TUNNEL_TOKEN") or "").strip()
    except Exception:
        t = ""
    if not t:
        t = os.environ.get("CF_TUNNEL_TOKEN", "").strip()
    return t.split()[-1] if t else ""

def start_cf_tunnel(token):
    subprocess.run(["pkill", "-9", "-f", "cloudflared"], check=False)
    if CF_PID.exists():
        CF_PID.unlink(missing_ok=True)
    time.sleep(1)
    cloudflared_bin = Path("/usr/local/bin/cloudflared")
    if not cloudflared_bin.exists():
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            str(cloudflared_bin),
        )
        cloudflared_bin.chmod(0o755)
    with CF_LOG.open("w") as cf_log:
        cf_proc = subprocess.Popen(
            [
                str(cloudflared_bin),
                "tunnel",
                "--no-autoupdate",
                "--protocol", "http2",
                "--edge-ip-version", "4",
                "run",
                "--url", "http://127.0.0.1:8188",
                "--token", token,
            ],
            stdin=subprocess.DEVNULL,
            stdout=cf_log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    CF_PID.write_text(str(cf_proc.pid))
    deadline = time.time() + 30
    while time.time() < deadline:
        if cf_proc.poll() is not None:
            print(CF_LOG.read_text(errors="replace")[-4000:])
            raise RuntimeError("Cloudflare Tunnel 启动失败，请检查 CF_TUNNEL_TOKEN 是否正确。")
        log_text = CF_LOG.read_text(errors="replace") if CF_LOG.exists() else ""
        if "Registered tunnel connection" in log_text:
            if "Updated to new configuration" not in log_text:
                time.sleep(1)
                log_text = CF_LOG.read_text(errors="replace") if CF_LOG.exists() else ""
            clean_log = log_text.replace('\"', '"')
            routes = re.findall(r'"hostname"\s*:\s*"([^"]+)"\s*,\s*"service"\s*:\s*"([^"]+)"', clean_log)
            if not routes:
                routes = [(h, s) for s, h in re.findall(r'"service"\s*:\s*"([^"]+)"\s*,\s*"hostname"\s*:\s*"([^"]+)"', clean_log)]
            if routes:
                for host, srv in routes:
                    print(f"[OK] Cloudflare Tunnel 已连接 (HTTP/2): https://{host} -> {srv}")
                    if "8188" not in srv:
                        print(f"[WARN] 检测到域名 {host} 在 Cloudflare 后台转发目标为 {srv}（非 ComfyUI 的 8188 端口）！")
                        print("       请前往 Cloudflare Zero Trust -> Networks -> Tunnels -> Public Hostname，将 Service URL 改为 http://127.0.0.1:8188")
            else:
                print("[OK] Cloudflare Tunnel 已连接 (HTTP/2)（请在 Cloudflare 绑定的域名访问 ComfyUI 或调用 API）。")
            return True
        time.sleep(1)
    print("[WARN] Cloudflare Tunnel 已在后台运行，详看 /content/cloudflared.log。")
    return True

cf_token = resolve_cf_token()
if not cf_token:
    raise RuntimeError("未检测到 CF_TUNNEL_TOKEN，请在 Colab 左侧 🔑 Secrets 中添加 CF_TUNNEL_TOKEN 并开启访问权限。")

LAUNCHER = Path("/content/indextts_comfy_userdata_compat.py")
LAUNCHER_SOURCE = r'''
"""Start pinned ComfyUI with narrow workflow aliases and Cloudflare compatibility."""
import os
import runpy
import sys
from pathlib import Path
from aiohttp import web


def add_workflow_aliases(manager, routes):
    originals = [route for route in routes
                 if getattr(route, "path", None) == "/userdata/{file}"
                 and getattr(route, "method", None) in {"GET", "POST", "DELETE"}]
    if {route.method for route in originals} != {"GET", "POST", "DELETE"}:
        raise RuntimeError("ComfyUI userdata routes changed; compatibility aliases were not installed.")
    for route in originals:
        async def workflow_alias(request, handler=route.handler):
            file = request.match_info.get("file", "")
            user_root = manager.get_request_user_filepath(request, None, create_dir=False)
            candidate = manager.get_request_user_filepath(request, file, create_dir=False)
            if not user_root or not candidate:
                return web.Response(status=403)
            workflow_directory = os.path.join(user_root, "workflows")
            if os.path.islink(workflow_directory):
                return web.Response(status=403)
            workflow_root = os.path.realpath(workflow_directory)
            try:
                within = os.path.commonpath((workflow_root, os.path.realpath(candidate))) == workflow_root
            except ValueError:
                within = False
            if not within or os.path.realpath(candidate) == workflow_root:
                return web.Response(status=403)
            return await handler(request)
        routes.route(route.method, "/userdata/{file:workflows/[^/]+}")(workflow_alias)


def install_cf_middleware():
    import middleware.cache_middleware as cm
    orig_cache_control = cm.cache_control

    @web.middleware
    async def cf_safe_middleware(request, handler):
        response = await orig_cache_control(request, handler)
        if request.method == "GET" and request.path == "/" and getattr(response, "status", None) == 200:
            import server
            inst = getattr(server.PromptServer, "instance", None)
            web_root = getattr(inst, "web_root", None) if inst else None
            index_file = Path(web_root) / "index.html" if web_root else None
            if index_file and index_file.is_file():
                html = index_file.read_text(encoding="utf-8", errors="replace")
                html = html.replace("<script ", '<script data-cfasync="false" ')
                new_resp = web.Response(text=html, content_type="text/html")
                for k, v in response.headers.items():
                    if k.lower() not in {"content-length", "content-type", "etag", "last-modified"}:
                        new_resp.headers[k] = v
                new_resp.headers["Cache-Control"] = "no-store, no-transform, must-revalidate"
                response = new_resp
        if isinstance(response, web.StreamResponse) and not isinstance(response, web.WebSocketResponse) and not response.prepared:
            if getattr(response, "status", None) == 200 and (
                request.path.startswith("/assets/")
                or request.path.endswith((".woff2", ".woff", ".ttf"))
            ):
                response.headers["Cache-Control"] = "public, max-age=604800, immutable, no-transform"
            else:
                cc = response.headers.get("Cache-Control", "")
                if "no-transform" not in cc:
                    response.headers["Cache-Control"] = f"{cc}, no-transform" if cc else "no-transform"
            origin = request.headers.get("Origin")
            if origin and response.headers.get("Access-Control-Allow-Origin") == "*":
                response.headers["Access-Control-Allow-Origin"] = origin
                response.headers["Vary"] = "Origin"
        return response

    cm.cache_control = cf_safe_middleware


def main():
    comfy_root = Path("/content/ComfyUI")
    if comfy_root.is_symlink() or not (comfy_root / "main.py").is_file() or not (comfy_root / "comfy/options.py").is_file():
        raise RuntimeError("ComfyUI source directory is missing or invalid.")
    os.chdir(comfy_root)
    sys.path.insert(0, str(comfy_root))
    sys.argv[0] = str(comfy_root / "main.py")
    import comfy.options
    comfy.options.enable_args_parsing()
    install_cf_middleware()
    from app.user_manager import UserManager
    original = UserManager.add_routes

    def compatible_routes(self, routes):
        original(self, routes)
        add_workflow_aliases(self, routes)

    UserManager.add_routes = compatible_routes
    runpy.run_path("main.py", run_name="__main__")


if __name__ == "__main__":
    main()
'''
launcher_updated = not LAUNCHER.exists() or LAUNCHER.read_text(errors="replace") != LAUNCHER_SOURCE
LAUNCHER.write_text(LAUNCHER_SOURCE)

if globals().get("INDEXTTS_INSTALLED_REVISIONS") != (globals().get("COMFY_REVISION"), globals().get("NODE_REVISION")):
    raise RuntimeError("请先运行第 2 格安装并检查环境。")

if ready() and launcher_updated:
    stop_idle_comfy()

if not ready():
    with LOG.open("w") as log:
        indextts_process = subprocess.Popen(
            [
                sys.executable, str(LAUNCHER),
                "--listen", get_listen_addrs(),
                "--port", "8188",
                "--output-directory", str(COMFY_DIR / "output"),
                "--enable-cors-header",
                "--enable-compress-response-body",
                "--disable-all-custom-nodes",
                "--whitelist-custom-nodes", "ComfyUI-IndexTTS-2_5",
                "--disable-partner-nodes",
            ],
            cwd=COMFY_DIR,
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    deadline = time.time() + 600
    while time.time() < deadline:
        if indextts_process.poll() is not None:
            print(LOG.read_text(errors="replace")[-8000:])
            raise RuntimeError("ComfyUI 启动失败，请查看上方日志。")
        if ready():
            print("[OK] ComfyUI 启动成功。")
            break
        time.sleep(2)
    else:
        raise TimeoutError("ComfyUI 10 分钟内未就绪。")
else:
    print("[OK] ComfyUI 已经在运行。")

registered = get_json("/object_info")
if not {"IndexTTS2_5Node", "EmoSettingsNode", "LoadAudio", "SaveAudioMP3"}.issubset(registered):
    print(LOG.read_text(errors="replace")[-8000:] if LOG.exists() else "")
    raise RuntimeError("缺少 IndexTTS-2.5 节点，请检查第 2 格安装日志：/content/indextts_comfyui.log。")

start_cf_tunnel(cf_token)
